In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import numpy as np

In [2]:
file_path = 'data.csv'

data = pd.read_csv(file_path)

print(data.head())
data.describe()


   Station                 Date  CO (ppm)  O3 (ppb)  NO (ppb)  NO2 (ppb)  \
0        1  2021-04-08 12:00:00      0.38       NaN      3.29       9.09   
1        1  2021-04-08 13:00:00      0.33       NaN      2.07       7.32   
2        1  2021-04-08 14:00:00      0.28       NaN      1.06       5.78   
3        1  2021-04-08 15:00:00      0.26       NaN      0.75       4.78   
4        1  2021-04-08 16:00:00      0.35       NaN      0.60       4.86   

   NOx (ppb)  SO2 (ppb)  PM10 (m3/mg)  PM2.5  (m3/mg)  
0      12.37      10.83           NaN             NaN  
1       9.39      10.87           NaN             NaN  
2       6.84      10.74           NaN            28.0  
3       5.53      10.10           NaN            23.0  
4       5.46      11.13           NaN            25.0  


,Station,CO (ppm),O3 (ppb),NO (ppb),NO2 (ppb),NOx (ppb),SO2 (ppb),PM10 (m3/mg),PM2.5 (m3/mg)
count,731060.000000,533887.000000,53181.000000,312403.000000,312633.000000,312615.000000,552463.000000,75756.000000,704109.000000
mean,7.348097,1.353163,37.903573,18.698435,20.287195,38.679525,6.133663,120.718499,40.315229
std,3.839181,2.310974,36.381593,56.128769,64.000128,70.590736,9.322242,144.514646,91.548734
min,1.000000,-2.980000,-315.270000,-692.480000,-134.500000,-684.720000,-6.090000,0.000000,-15.000000
25%,4.000000,0.370000,7.820000,1.680000,8.900000,12.160000,1.880000,66.840000,17.000000
50%,8.000000,0.830000,28.150000,5.110000,17.380000,27.300000,3.820000,95.140000,25.580000
75%,11.000000,1.710000,57.050000,18.830000,28.010000,46.120000,7.640000,143.542500,39.400000
max,13.000000,849.110000,788.620000,10730.000000,19615.590000,26573.380000,947.150000,23829.740000,3525.720000


In [3]:
df = data[['Station', 'Date', 'NO2 (ppb)']]
print(df.head())
df.describe()

   Station                 Date  NO2 (ppb)
0        1  2021-04-08 12:00:00       9.09
1        1  2021-04-08 13:00:00       7.32
2        1  2021-04-08 14:00:00       5.78
3        1  2021-04-08 15:00:00       4.78
4        1  2021-04-08 16:00:00       4.86


,Station,NO2 (ppb)
count,731060.000000,312633.000000
mean,7.348097,20.287195
std,3.839181,64.000128
min,1.000000,-134.500000
25%,4.000000,8.900000
50%,8.000000,17.380000
75%,11.000000,28.010000
max,13.000000,19615.590000


In [4]:
df.rename(columns={'NO2 (ppb)':'NO2'}, inplace = True)
data = df

/var/folders/q2/f903sqhx2x111k92n1m9s0tr0000gn/T/ipykernel_57537/3242758560.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'NO2 (ppb)':'NO2'}, inplace = True)


In [5]:
# Sort by station and date

data = data.sort_values(by=['Station', 'Date'])
print(data.head())

       Station                 Date    NO2
14545        1  2018-08-14 12:00:00  16.76
14546        1  2018-08-14 13:00:00  17.45
14547        1  2018-08-14 14:00:00  16.70
14548        1  2018-08-14 15:00:00  16.84
14549        1  2018-08-14 16:00:00  17.13


In [6]:
# Count each station's data

station_counts = data['Station'].value_counts().reindex(range(1, 14), fill_value=0)
print("Number of rows for each Station (1 to 13):")
print(station_counts)

Number of rows for each Station (1 to 13):
Station
1     60825
2     56591
3     38438
4     38108
5     72802
6     53488
7     21827
8     70309
9     62637
10    40565
11    79937
12    80671
13    54862
Name: count, dtype: int64


In [7]:
# Calculating min and max date for all stations

station_dates = data.groupby('Station')['Date'].agg(['min', 'max']).reindex(range(1, 14))
print("\nMin and Max dates for each Station:")
print(station_dates)


Min and Max dates for each Station:
                         min                  max
Station                                          
1        2018-08-14 12:00:00  2023-08-23 09:59:00
2        2018-08-14 12:00:00  2023-08-23 10:04:00
3        2023-04-09 14:42:00  2023-08-21 23:59:00
4        2023-04-09 16:57:00  2023-08-22 00:00:00
5        2018-03-21 02:00:00  2023-08-23 10:05:00
6        2018-08-14 11:00:00  2023-08-23 09:58:00
7        2023-05-31 00:00:00  2023-08-21 23:59:00
8        2018-08-14 12:00:00  2023-08-21 23:58:00
9        2018-08-14 12:00:00  2023-08-23 10:15:00
10       2023-03-21 00:00:00  2023-08-21 18:14:00
11       2018-03-21 02:00:00  2023-08-21 23:57:00
12       2018-03-21 02:00:00  2023-08-23 10:08:00
13       2018-03-21 02:00:00  2023-07-18 12:26:00


In [8]:
# Only keep data from before 1402

data['Date'] = pd.to_datetime(data['Date'])
cutoff_date = pd.to_datetime('2023-03-21')
df_filtered = data[data['Date'] < cutoff_date]

station_dates = df_filtered.groupby('Station')['Date'].agg(['min', 'max']).reindex(range(1, 14))
print("\nMin and Max dates for each Station:")
print(station_dates)


Min and Max dates for each Station:
                        min        max
Station                               
1       2018-08-14 12:00:00 2023-03-20
2       2018-08-14 12:00:00 2023-03-20
3                       NaT        NaT
4                       NaT        NaT
5       2018-03-21 02:00:00 2023-03-20
6       2018-08-14 11:00:00 2023-03-20
7                       NaT        NaT
8       2018-08-14 12:00:00 2023-03-20
9       2018-08-14 12:00:00 2023-03-20
10                      NaT        NaT
11      2018-03-21 02:00:00 2023-03-20
12      2018-03-21 02:00:00 2023-03-20
13      2018-03-21 02:00:00 2023-03-20


In [9]:
data = df_filtered
print(data.head())

       Station                Date    NO2
14545        1 2018-08-14 12:00:00  16.76
14546        1 2018-08-14 13:00:00  17.45
14547        1 2018-08-14 14:00:00  16.70
14548        1 2018-08-14 15:00:00  16.84
14549        1 2018-08-14 16:00:00  17.13


In [10]:
# Count each station's remaining data

station_counts = data['Station'].value_counts().reindex(range(1, 14), fill_value=0)
print("Number of rows for each Station (1 to 13):")
print(station_counts)

Number of rows for each Station (1 to 13):
Station
1     34862
2     36077
3         0
4         0
5     43683
6     31749
7         0
8     34829
9     33497
10        0
11    43683
12    43683
13    43683
Name: count, dtype: int64


In [11]:
# Remove stations 3, 4, 7, 10

used_stations = [1, 2, 5, 6, 8, 9, 11, 12, 13]
new_labels = range(1, len(used_stations) + 1)

station_mapping = {old: new for old, new in zip(used_stations, new_labels)}

data['Station'] = data['Station'].map(station_mapping)

print(data.head())

       Station                Date    NO2
14545        1 2018-08-14 12:00:00  16.76
14546        1 2018-08-14 13:00:00  17.45
14547        1 2018-08-14 14:00:00  16.70
14548        1 2018-08-14 15:00:00  16.84
14549        1 2018-08-14 16:00:00  17.13


In [12]:
print(data['Station'].nunique())

9


In [13]:
# Only keep data that every station has

data = data.sort_values(by=['Date'])
valid_dates = data.groupby('Date').filter(lambda x: x['Station'].nunique() == 9)['Date'].unique()
data_filtered = data[data['Date'].isin(valid_dates)]
data = data_filtered
data = data.sort_values(by = 'Station')

In [14]:
# Drop duplicate data

data = data.drop_duplicates(subset=['Station', 'Date'], keep='first')

In [15]:
# Count each stations's remaining data

station_counts = data['Station'].value_counts().reindex(range(1, 10), fill_value=0)
print("Number of rows for each Station (1 to 9):")
print(station_counts)

Number of rows for each Station (1 to 9):
Station
1    23470
2    23470
3    23470
4    23470
5    23470
6    23470
7    23470
8    23470
9    23470
Name: count, dtype: int64


In [16]:
na_counts_per_station = data.groupby('Station')['NO2'].apply(lambda x: x.isna().sum()).reset_index(name='na_count')
print(na_counts_per_station)

   Station  na_count
0        1        67
1        2       117
2        3     23470
3        4       105
4        5         0
5        6        82
6        7     23470
7        8     23470
8        9     23470


In [17]:
# Remove stations 3, 7, 8, 9

data = data[~data['Station'].isin([3, 7, 8, 9])]

used_stations = [1, 2, 4, 5, 6]
new_labels = range(1, len(used_stations) + 1)

station_mapping = {old: new for old, new in zip(used_stations, new_labels)}

data['Station'] = data['Station'].map(station_mapping)

print(data.head())

       Station                Date    NO2
25675        1 2019-12-27 16:00:00   7.52
12167        1 2022-09-18 01:00:00  32.10
18576        1 2019-02-11 15:00:00  16.88
22631        1 2019-08-22 08:00:00   0.00
54066        1 2020-04-14 11:00:00   3.69


In [18]:
print(data['Station'].nunique())

5


In [19]:
na_counts_per_station = data.groupby('Station')['NO2'].apply(lambda x: x.isna().sum()).reset_index(name='na_count')
print(na_counts_per_station)

   Station  na_count
0        1        67
1        2       117
2        3       105
3        4         0
4        5        82


In [20]:
# fill all NAs
data = data.ffill().bfill()

In [21]:
data.to_csv('./NO2_preprocessed_data.csv')

In [ ]:
import numpy as np

def prepare_data(r=12, b=5):
  pm25_data = data['PM2.5'].values

  total_data = station_counts[1]

  inputs = []
  outputs = []

  for i in range(0, total_data - r - b + 1, r//4):
      input = []
      output = []
      for j in range(9):
        input.append(pm25_data[j * total_data + i:j * total_data + i + r])
        output.append(pm25_data[j * total_data + i + r:j * total_data + i + r + b])
      inputs.append(input)
      outputs.append(output)

  inputs = np.array(inputs)
  outputs = np.array(outputs)
  return inputs, outputs

In [ ]:
from sklearn.model_selection import train_test_split

inputs, outputs = prepare_data()
print(inputs.shape, outputs.shape)

train_ratio = 0.6
val_ratio = 0.2
test_ratio = 0.2

X_train, X_temp, y_train, y_temp = train_test_split(inputs, outputs, test_size=(val_ratio + test_ratio), shuffle=False)

val_size = val_ratio / (val_ratio + test_ratio)
test_size = test_ratio / (val_ratio + test_ratio)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=test_size, shuffle=False)

(7818, 9, 12) (7818, 9, 5)
